# 05 — Evaluation & Costs

Quantify the qualitative impressions from notebook 04.

**Metrics:**
- **Faithfulness** — does the answer stick to retrieved context? (RAGAS)
- **Answer relevance** — does it actually answer the question? (RAGAS)
- **Context recall / precision** — did retrieval surface the right context? (RAGAS)
- **Latency** — wall-clock per query
- **Cost** — token usage × model price

In [ ]:
from rag_compare.agentic_rag import AgenticRagPipeline
from rag_compare.common import load_corpus
from rag_compare.eval import run_eval
from rag_compare.hybrid_rag import HybridRagPipeline
from rag_compare.kg_rag import KGRagPipeline

docs = load_corpus()
kg = KGRagPipeline.build(docs)
hybrid = HybridRagPipeline.build(docs)
agent = AgenticRagPipeline.build(hybrid=hybrid, kg=kg)

In [ ]:
kg_eval = run_eval(kg, "kg_rag")
hybrid_eval = run_eval(hybrid, "hybrid_rag")
agent_eval = run_eval(agent, "agentic_rag")

kg_eval.df, hybrid_eval.df, agent_eval.df

## Add RAGAS metrics

```python
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from datasets import Dataset

ds = Dataset.from_pandas(eval_result.df.rename(columns={"expected": "ground_truth"}))
scores = evaluate(ds, metrics=[faithfulness, answer_relevancy, context_recall, context_precision])
```

Note: RAGAS needs both the answer and the retrieved contexts. Update the eval harness to also return `contexts` from each pipeline (use the underlying retrievers, not just `query()`).

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "kg_rag": kg_eval.summary(),
    "hybrid_rag": hybrid_eval.summary(),
    "agentic_rag": agent_eval.summary(),
})
summary

## Cost estimation

Wrap the LLM client with a token counter (LlamaIndex has `TokenCountingHandler`) and multiply by your provider's per-token rate. Plot `$/query` × `quality` to see the Pareto frontier — usually:
- Hybrid is cheap and competitive
- KG-RAG has high index cost but cheap queries
- Agentic is expensive per query but wins on hard questions